# Hamilton Liquid Classes

This notebook demonstrates how to use Hamilton liquid classes with the STAR.
"Liquid classes" are sets of predefined parameters that describe a specific liquid transfer
operation (aspirate + dispense). While it is possible to control all parameters explicitly,
using "Hamilton liquid classes" is the historical way many people are used to doing this in
VENUS.

PyLabRobot has imported many Hamilton liquid classes from VENUS. In this notebook we will
show how to use these classes.

## Setup

Use `chatterbox=True` to simulate without connecting to a real Hamilton STAR robot.
Remove `chatterbox=True` (or pass `chatterbox=False`) to run on real hardware.

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star import STARLet

star = STARLet(chatterbox=True)
# star = STARLet()  # for real hardware
await star.setup()

In [ ]:
from pylabrobot.resources import (
  TIP_CAR_480_A00,
  PLT_CAR_L5AC_A00,
  Cor_96_wellplate_360ul_Fb,
  hamilton_96_tiprack_1000uL_filter,
)

tip_car = TIP_CAR_480_A00(name="tip carrier")
tip_car[0] = hamilton_96_tiprack_1000uL_filter(name="tips_01")
star.deck.assign_child_resource(tip_car, rails=3)

plt_car = PLT_CAR_L5AC_A00(name="plate carrier")
plt_car[0] = Cor_96_wellplate_360ul_Fb(name="plate_01")
star.deck.assign_child_resource(plt_car, rails=15)

### Picking up tips for the rest of the notebook

In [ ]:
tiprack = star.deck.get_resource("tips_01")
plate = star.deck.get_resource("plate_01")

await star.pip.pick_up_tips(tiprack["A1:C1"])

## Using a predefined Hamilton liquid class

Pass a predefined Hamilton liquid class to `star.pip.aspirate` via
{class}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.STARPIPBackend.AspirateParams`.
This will use the parameters defined in the liquid class for the aspirate operation.

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star.pip_backend import STARPIPBackend
from pylabrobot.liquid_handling.liquid_classes.hamilton.star import HighVolumeFilter_Water_DispenseSurface_Part

await star.pip.aspirate(
  plate["A1:C1"],
  vols=[100.0, 50.0, 200.0],
  backend_params=STARPIPBackend.AspirateParams(
    hamilton_liquid_classes=[HighVolumeFilter_Water_DispenseSurface_Part] * 3,
  ),
)

## Using a different Hamilton liquid class for each channel

You can pass a list of different Hamilton liquid classes. They will correspond to the
channels in the order they are specified.

In [ ]:
from pylabrobot.liquid_handling.liquid_classes.hamilton.star import (
  HighVolumeFilter_Water_DispenseSurface_Part,
  HighVolumeFilter_EtOH_DispenseJet,
  HighVolumeFilter_DMSO_AliquotDispenseJet_Part,
)

await star.pip.aspirate(
  plate["A1:C1"],
  vols=[100.0, 50.0, 200.0],
  backend_params=STARPIPBackend.AspirateParams(
    hamilton_liquid_classes=[
      HighVolumeFilter_Water_DispenseSurface_Part,
      HighVolumeFilter_EtOH_DispenseJet,
      HighVolumeFilter_DMSO_AliquotDispenseJet_Part,
    ],
  ),
)

## Using custom Hamilton liquid classes

It is also possible to define your own Hamilton liquid classes. This is useful if you want
to use a specific set of parameters that are not available in the predefined classes.

The example below is based on `HighVolumeFilter_Water_DispenseSurface_Part`, but you can
easily modify the parameters to suit your needs.

In [ ]:
from pylabrobot.liquid_handling.liquid_classes.hamilton import HamiltonLiquidClass

my_custom_hamilton_liquid_class = HamiltonLiquidClass(
  curve={
  500.0: 518.3,
  50.0: 56.3,
  0.0: 0.0,
  100.0: 108.3,
  20.0: 23.9,
  1000.0: 1028.5,
  200.0: 211.0,
  10.0: 12.7,
  },
  aspiration_flow_rate=250.0,
  aspiration_mix_flow_rate=120.0,
  aspiration_air_transport_volume=0.0,
  aspiration_blow_out_volume=0.0,
  aspiration_swap_speed=2.0,
  aspiration_settling_time=1.0,
  aspiration_over_aspirate_volume=5.0,
  aspiration_clot_retract_height=0.0,
  dispense_flow_rate=120.0,
  dispense_mode=4.0,
  dispense_mix_flow_rate=1.0,
  dispense_air_transport_volume=30.0,
  dispense_blow_out_volume=0.0,
  dispense_swap_speed=2.0,
  dispense_settling_time=1.0,
  dispense_stop_flow_rate=5.0,
  dispense_stop_back_volume=0.0,
)

await star.pip.aspirate(
  plate["A1:C1"],
  vols=[100.0, 50.0, 200.0],
  backend_params=STARPIPBackend.AspirateParams(
  hamilton_liquid_classes=[my_custom_hamilton_liquid_class] * 3,
  ),
)

## Teardown

In [ ]:
await star.stop()